# Outlier robustness: balanced vs unbalanced barycenter

10 distributions in 10-D, each a Gaussian mixture with 4 inlier modes (mass 0.94) and 4 far outlier
modes (mass 0.06). The uniform-weight barycenter is computed with balanced OT and with unbalanced
OT (tau=1): the UOT barycenter recovers the inlier modes, the balanced one is dragged toward the
outliers.

In [ ]:
import os
import sys

import numpy as np
import matplotlib.pyplot as plt


def _bootstrap():
    here = os.path.abspath(os.path.dirname(__file__)) if "__file__" in globals() else os.getcwd()
    # this folder (its helper modules) + tools/ (the shared `_repo.py` path resolver)
    for d in (here, os.path.abspath(os.path.join(here, os.pardir, os.pardir, "tools"))):
        if d not in sys.path:
            sys.path.insert(0, d)
    import _repo
    _repo.add_paths()
    return _repo


P = _bootstrap()
REPO_ROOT = P.REPO
import uotreg as U
from uotreg.config import ModelConfig, TrainConfig, UOTConfig
from uotreg import outlier_sim as od

# ----------------------------------------------------------------------------- SMOKE
# 1 = small and fast: runs end to end on a laptop. **NOT the paper's numbers.**
# 0 = the settings used in the paper.
SMOKE = 1

DEVICE = "auto"

## 1. Generate the 10 mixtures and look at the first 2 dims

In [ ]:
clouds = od.generate_outlier_gmms(num=10, n=2000, seed=0)
samplers = U.samplers_from_arrays(clouds, device=DEVICE)
weights = [1.0 / len(clouds)] * len(clouds)          # uniform barycenter weights

pooled = np.concatenate(clouds, axis=0)
plt.figure(figsize=(6, 5), dpi=120)
plt.scatter(pooled[:, 0], pooled[:, 1], s=4, alpha=0.25, c="lightgrey", label="data")
plt.scatter(od.INLIER_MEANS[:, 0], od.INLIER_MEANS[:, 1], marker="*", s=200, c="tab:blue", label="inlier modes")
plt.scatter(od.OUTLIER_MEANS[:, 0], od.OUTLIER_MEANS[:, 1], marker="X", s=120, c="tab:red", label="outlier modes")
plt.legend(); plt.title("10-D outlier mixture (dims 1-2)"); plt.show()

## 2. Train both barycenters (matched config)

In [ ]:
model = ModelConfig(dim=od.DIM, gen_hidden=256, gen_layers=4, gen_dropout=0.05,
                    map_hidden=100, map_layers=3, pot_hidden=100, pot_layers=3, dropout=0.05)
train = TrainConfig(outer_iters=(4 if SMOKE else 41), d_iters=(10 if SMOKE else 50),
                    t_iters=(3 if SMOKE else 10), g_iters=(10 if SMOKE else 50),
                    batch_size=64, batch_size_g=128, lr_map=3e-4, lr_pot=3e-4, lr_gen=1e-4,
                    weight_decay_td=1e-10, weight_decay_gen=1e-8, device=DEVICE,
                    verbose=True, log_every=10, seed=0)

est_bal = U.DistributionEstimator(model=model, train=train, uot=UOTConfig(relaxation="balanced"))
est_bal.fit(samplers, weights=weights, init="gaussian", init_kwargs={"iters": (500 if SMOKE else 10000), "gaussian_scale": 6.0})

est_uot = U.DistributionEstimator(model=model, train=train, uot=UOTConfig(relaxation="one-sided", tau=1.0))
est_uot.fit(samplers, weights=weights, init="gaussian", init_kwargs={"iters": (500 if SMOKE else 10000), "gaussian_scale": 6.0})

## 3. Compare: visual + robustness metrics

In [ ]:
X_bal, X_uot = est_bal.sample(2000), est_uot.sample(2000)

fig, axes = plt.subplots(1, 2, figsize=(13, 5), dpi=120)
for ax, X, name in [(axes[0], X_bal, "balanced OT"), (axes[1], X_uot, "UOT (tau=1)")]:
    ax.scatter(pooled[:, 0], pooled[:, 1], s=4, alpha=0.15, c="lightgrey")
    ax.scatter(X[:, 0], X[:, 1], s=8, alpha=0.6, c="tab:purple", label="barycenter")
    ax.scatter(od.INLIER_MEANS[:, 0], od.INLIER_MEANS[:, 1], marker="*", s=200, c="tab:blue")
    ax.scatter(od.OUTLIER_MEANS[:, 0], od.OUTLIER_MEANS[:, 1], marker="X", s=120, c="tab:red")
    ax.set_title(name); ax.legend(fontsize=8)
fig.tight_layout(); plt.show()

print(f"{'method':14s}{'outlier contamination':>24s}{'mean dist to inlier mode':>28s}")
for name, X in [("balanced OT", X_bal), ("UOT (tau=1)", X_uot)]:
    print(f"{name:14s}{od.outlier_contamination(X):24.3f}{od.inlier_mode_distance(X):28.3f}")
print("\n(lower = more robust; UOT should have ~0 contamination and smaller inlier distance)")